# ಪಾಠ 08 - ಬಹು-ಏಜೆಂಟ್ ವಿನ್ಯಾಸ ಮಾದರಿ


## ಸಿದ್ಧತೆ


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ಏಕೆ ಮಲ್ಟಿ-ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳು?

ಪ್ರವಾಸ ಯೋಜನೆ ಹಾಗು ಇತರೆ ನಾಗರಿಕ ಕಾರ್ಯಗಳು ಹಲವಾರು ವಿವಿಧ ತಜ್ಞತೆಗಳನ್ನು ಒಳಗೊಂಡಿರುತ್ತವೆ — ಲಾಜಿಸ್ಟಿಕ್ಸ್, ಸ್ಥಳೀಯ ತಿಳಿವು, ಬಜೆಟಿಂಗ್ ಇತ್ಯಾದಿ. ಎಲ್ಲವನ್ನೂ ನಿಭಾಯಿಸಲು ಒಬ್ಬ ಏಜೆಂಟ್ ಪ್ರಯತ್ನಿಸಿದರೆ ಅದನ್ನು ನಿರ್ವಹಿಸುವುದು ವೇಗವಾಗಿ ಕಷ್ಟಕಾರಿಯಾದೀತು.

ಮಲ್ಟಿ-ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳು ಇದನ್ನು **ವಿಶೇಷೀಕರಣ** ಮೂಲಕ ಪರಿಹರಿಸುತ್ತವೆ: ಪ್ರತಿ ಏಜೆಂಟ್ ಒಂದೇ ತಜ್ಞತೆಗೆ ಗಮನಹರಿಸಿ, ಸಾಮಾನ್ಯ ವ್ಯಕ್ತಿಗಿಂತ ಉತ್ತಮ ಗುಣಮಟ್ಟದ ಫಲಿತಾಂಶಗಳನ್ನು ನೀಡುತ್ತದೆ. ಇವು **ವಿಸ್ತರಣೆ ಸಾಮರ್ಥ್ಯ** (scalability) ಇದ್ದಂತೆ ಮಾಡುತ್ತವೆ — ಹೊಸ ಏಜೆಂಟ್ ಗಳನ್ನು (ಉದಾಹರಣೆಗೆ, ವಿಮಾನ ತಜ್ಞ, ರೆಸ್ಟೋರೆಂಟ್ ವಿಮರ್ಶಕ) ಈಗಿನ ಕಾರ‍್ಯಕ್ರಮವನ್ನು ಪುನರೆഴೆಯದೇ ಸೇರಿಸಬಹುದು. ಏಜೆಂಟ್ ಗಳು ಸಂರಚಿತ ಪೈಪ್‌ಲೈನ್ ಮೂಲಕ ಸಂಯೋಜಿಸಲ್ಪಡುವಾಗ, ಒಂದು ಏಜೆಂಟ್ ರಿಂದ ಮುಂದಿನ ಏಜೆಂಟ್ ಗೆ ಪರಿಚ.Context ಅನ್ನು ಹಸ್ತಾಂತರಿಸುತ್ತವೆ.


## ವಿಶಿಷ್ಠ ಏಜನ್ಸಿಗಳನ್ನು ರಚಿಸುವುದು


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ಕ್ರಮಬದ್ಧ ಕಾರ್ಯಪ್ರವಾಹವನ್ನು ನಿರ್ಮಿಸುವದು

`WorkflowBuilder` ನಿಮಗೆ ಏಜೆಂಟ್‌ಗಳನ್ನು ನಿರ್ದೇಶಿತ ಗ್ರಾಫ್‌ನಲ್ಲಿ ಸಂಪರ್ಕಿಸಲು ಅವಕಾಶ ನೀಡುತ್ತದೆ. ಇಲ್ಲಿ ನಾವು ಒಂದು ಸರಳ ಎರಡು ಹಂತದ ಪೈಪ್‌ಲೈನ್ ಅನ್ನು ರಚಿಸುತ್ತೇವೆ: **TravelPlanner** ಪ್ರಯಾಣ ಕಾರ್ಯಕ್ರಮವನ್ನು मसೂದು ಮಾಡುತ್ತದೆ, ನಂತರ **TravelConcierge** ಅವುಗಳನ್ನು ಪರಿಶೀಲಿಸಿ ಸುಧಾರಿಸುತ್ತದೆ.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ವರ್ಕ್‌ಫ್ಲೋಗೆ ಇನ್ನಷ್ಟು ಏಜೆಂಟುಗಳನ್ನು ಸೇರಿಸುವುದು

ಮಲ್ಟಿ-ಏಜೆಂಟ್ ಮಾದರಿಯ ಪ್ರಮುಖ ಲಾಭಗಳಲ್ಲಿ ಒಂದೆಂದರೆ ಅದನ್ನು ವಿಸ್ತರಿಸಲು ಎಷ್ಟು ಸುಲಭವೆಂಬುದು. ಕೆಳಗೆ ನಾವು **BudgetReviewer** ಎಂಬ ಏಜೆಂಟ್ ಅನ್ನು ಸೇರಿಸುತ್ತಿದ್ದೇವೆ, ಇದು ಯೋಜನೆಯನ್ನು ಪ್ರಯಾಣಿಕನ ಬಜೆಟ್‌ಗೆ ವಿರುದ್ಧವಾಗಿ ಪರಿಶೀಲಿಸುತ್ತದೆ, ವೆಚ್ಚವನ್ನು ಮೀರಿಸುವ ಸಾಧ್ಯತೆ ಇರುವ ಐಟಂಗಳನ್ನು ಗುರುತಿಸುತ್ತದೆ ಮತ್ತು ಹಣ ಉಳಿಸುವ ಪರ್ಯಾಯಗಳನ್ನು ಸೂಚಿಸುತ್ತದೆ. ವರ್ಕ್‌ಫ್ಲೋ ಈಗ ಕ್ರಮವಾಗಿ ಮೂರು ಏಜೆಂಟ್‍ಗಳನ್ನು ನಡಿಸುತ್ತದೆ:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಕೆಳಕಂಡವುಗಳನ್ನhow ಕಲಿತಿರಿ:

1. **ವಿಶೇಷ ಆಗೆಂಟ್‌ಗಳನ್ನು ಸೃಷ್ಟಿಸುವುದು** — ಪ್ರತಿ ಒಬ್ಬರಿಗೂ ಗುರಿಪಡಿಸಿದ ಪಾತ್ರವಿದೆ (ಯೋಜನೆ, ಕಾನ್ಸಿಯರ್ಜ್, ಬಜೆಟ್ ವಿಮರ್ಶೆ).
2. **`WorkflowBuilder` ಮತ್ತು `add_edge` ಬಳಸಿ ಆಗೆಂಟ್‌ಗಳನ್ನೊದಗಿಸುವ ಕ್ರಮ ಬದ್ಧ ಪ್ರಕ್ರಿಯೆಗೆ**.
3. **ಬಹು-ಆಗೆಂಟ್ ಪೈಪ್ಲೈನ್‌ನಿಂದ ಔಟ್‌ಪುಟ್ ಅನ್ನು ಸ್ಟ್ರೀಮ್ ಮಾಡುವುದು**, ಯಾವ ಆಗೆಂಟ್ ಮಾತನಾಡುತ್ತಿರೋದು ಟ್ರ್ಯಾಕ್ ಮಾಡುವುದು.
4. **ಪ್ರಗತಿಯಲ್ಲಿರುವಾಗಲೇ ಹೊಸ ಆಗೆಂಟ್‌ಗಳನ್ನು ಸರಣಿಗೆ ಸೇರಿಸಿ ಕಾರ್ಯಪಟುವನ್ನು ವಿಸ್ತರಿಸುವುದು**.

ಬಹು-ಆಗೆಂಟ್ ವಿನ್ಯಾಸ ಮಾದರಿಯು ಪ್ರತಿ ಆಗೆಂಟ್ ಅನ್ನು ಸರಳವಾಗಿಡುತ್ತದೆ ಮತ್ತು ಒಬ್ಬರಿಗೆ ಸಾಧ್ಯವಿಲ್ಲದಷ್ಟು ಸಮೃದ್ಧ ಮತ್ತು ಸ್ವತಃ ಪರಿಶೀಲಿತ ಫಲಿತಾಂಶಗಳನ್ನು ಉತ್ಪಾದಿಸುತ್ತದೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ನಿಹಿತತ್ವ**:  
ಈ ದಾಖಲೆಯನ್ನು AI ಭಾಷಾಂತರ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಭಾಷಾಂತರಿಸಲಾಗಿದೆ. ನಾವು ಸರಳತೆಯ ಬಗ್ಗೆ ಪ್ರಯತ್ನಿಸುವಾಗ, ಸ್ವಯಂಚಾಲಿತ ಭಾಷಾಂತರಗಳಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ಅಸತ್ಯತೆಗಳು ಇರಬಹುದೆಂದು ದಯವಿಟ್ಟು ಗಮನಿಸಿ. ಮೂಲ ಭಾಷೆಯ ಮೂಲ ಡಾಕ್ಯುಮೆಂಟ್ ಅನ್ನು ಅಧಿಕೃತ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ ವೃತ್ತಿಪರ ಮಾನವ ಭಾಷಾಂತರವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಭಾಷಾಂತರವನ್ನು ಬಳಸಿ ಉಂಟಾಗಬಹುದಾದ ಯಾವುದೇ ಅರ್ಥಘಟನೆಯ ತಪ್ಪುಗಳಿಗೆ ಅಥವಾ ತಪ್ಪಿತಸ್ಥತೆಗಳಿಗೆ ನಾವು ಜವಾಬ್ದಾರಿಯಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
